# 02 — EDA: Pattern di Engagement

> **Notebook 02 di 7** | Learning Retention Analytics  
> Seconda analisi esplorativa: come interagiscono gli studenti con la piattaforma e cosa rivelano i pattern di engagement?

## Scopo e ambito

Questo notebook è il **secondo di due notebook EDA**. Il Notebook 01 ha profilato la *base studenti* (dati demografici, esiti, panoramica dei corsi). Qui ci spostiamo sull'**engagement comportamentale** — i pattern di clickstream che riflettono cosa gli studenti effettivamente *fanno* sulla piattaforma.

**Cosa copre questo notebook:**
- Ghost student — iscrizioni con zero attività VLE
- Distribuzione delle metriche di engagement precoce (primi 28 giorni)
- Relazione dose-risposta tra decile di engagement ed esito
- Traiettoria di engagement giornaliero — quando si apre il divario?
- Tipologia di engagement — binge vs. steady
- Diversità dei tipi di attività — quali risorse usano gli studenti?
- Profili di engagement a livello di corso
- Riepilogo della correlazione engagement-esito

**Cosa viene dopo:**
- **Notebook 03** (`03_bq1_dropout_timing.ipynb`): dove e quando abbandonano gli studenti? (BQ1)

**Collegamento alle business question:**  
Questa EDA non risponde direttamente a BQ1–BQ5. Piuttosto, costruisce la **base comportamentale** per BQ2 (segnali precoci che predicono l'abbandono), BQ3 (demografica vs. comportamento) e BQ5 (interventi azionabili). Pensalo come *mappare il panorama comportamentale prima di testare ipotesi specifiche*.

> **Trasferibilità metodologica:** I pattern di engagement esplorati qui — ghost user, segnali della finestra di onboarding, utilizzo binge vs. steady, diversità delle attività — sono direttamente portabili alla retention SaaS, al churn da abbonamento e all'engagement delle app fitness. Il *dominio* è l'istruzione; il *framework analitico* è la product analytics.

## Indice

1. [Setup dell'ambiente](#1.-Setup-dell'ambiente)
2. [Ghost Student — Zero Engagement](#2.-Ghost-Student-—-Zero-Engagement)
3. [Distribuzioni dell'engagement precoce](#3.-Distribuzioni-dell'engagement-precoce)
4. [Decili di engagement vs. esito](#4.-Decili-di-engagement-vs.-esito)
5. [Traiettoria di engagement giornaliero](#5.-Traiettoria-di-engagement-giornaliero)
6. [Tipologia di engagement — Binge vs. Steady](#6.-Tipologia-di-engagement-—-Binge-vs.-Steady)
7. [Diversità dei tipi di attività](#7.-Diversità-dei-tipi-di-attività)
8. [Profili di engagement a livello di corso](#8.-Profili-di-engagement-a-livello-di-corso)
9. [Matrice di correlazione engagement-esito](#9.-Matrice-di-correlazione-engagement-esito)
10. [Conclusioni e prossimi passi](#10.-Conclusioni-e-prossimi-passi)

---

**Prerequisiti:**
- La pipeline ETL deve essere stata eseguita: `python -m run_pipeline`
- Il database DuckDB in `data/db/oulad.duckdb` deve contenere tutte e 5 le viste analitiche

**Dataset:** Open University Learning Analytics Dataset (OULAD) — ~32K studenti, 7 corsi, clickstream comportamentale completo. Licenza: CC-BY 4.0.

## 1. Setup dell'ambiente

Configuriamo import, default di visualizzazione e funzioni helper riutilizzabili.

**Note tecniche per il lettore:**
- I notebook risiedono in `notebooks/` ma i moduli del progetto sono in `src/` nella root del progetto. Aggiungiamo la project root a `sys.path` affinché `from src.config import ...` funzioni. La regola del linter `E402` (import non in cima al file) è soppressa per i notebook in `pyproject.toml`.
- Tutte le query al database passano da `src.db.connection.execute_query()` — il layer di astrazione DB del progetto. Questo restituisce un `pandas.DataFrame` e assicura che non si chiami mai `duckdb.connect()` direttamente (vedi ADR-003).
- Le figure vengono salvate in `reports/figures/` a 150 DPI. Poiché `nbstripout` rimuove gli output del notebook prima del commit, i PNG salvati sono il record visivo persistente.

In [ ]:
# --- Path setup ---
# Notebooks live in notebooks/ but project modules are at the project root.
# Adding the root to sys.path lets us import from src/ as if running from root.
# We search upward for pyproject.toml instead of assuming cwd is always notebooks/,
# so the notebook works regardless of where the kernel is launched from
# (JupyterLab, VS Code, Cursor, repo root, etc.).
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from start until we find pyproject.toml (the repo root marker)."""
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate project root: no pyproject.toml found in '{start}' "
        "or any parent directory. Run the notebook from within the repository."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Standard library ---
import logging
import warnings

# --- Third-party ---
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import seaborn as sns

# --- Project modules ---
from src.config import FIGURES_DIR
from src.db.connection import execute_query

# --- Configuration ---
# Suppress noisy warnings in notebook output; errors still surface
warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.WARNING)

# --- Visualization defaults ---
# Consistent style across all project notebooks
sns.set_theme(style='whitegrid', font_scale=1.1)

# Semantic color palette: one color per outcome category
PALETTE_OUTCOME = {
    'Pass': '#4C72B0',        # blue — neutral positive
    'Distinction': '#55A868', # green — strong positive
    'Fail': '#C44E52',        # red — negative
    'Withdrawn': '#8172B3',   # purple — departed (distinct from failed)
}
# Outcome label constants — single source of truth for the binary labels
# used in SQL output mapping, palette keys, and plot iterations
LABEL_COMPLETED = 'Completed'
LABEL_NOT_COMPLETED = 'Not completed'
# Binary version: completed (1) vs not completed (0)
PALETTE_BINARY = {1: '#55A868', 0: '#C44E52'}
LABEL_BINARY = {1: LABEL_COMPLETED, 0: LABEL_NOT_COMPLETED}
# Label-keyed palette for seaborn when x-axis uses mapped string categories
PALETTE_BINARY_LABELS = {LABEL_COMPLETED: '#55A868', LABEL_NOT_COMPLETED: '#C44E52'}
# Sequential palette for heatmaps and continuous scales
PALETTE_SEQUENTIAL = 'YlOrRd'
# Shared axis labels — avoids cross-cell string literal duplication
LABEL_NUM_ENROLLMENTS = 'Number of enrollments'
LABEL_ACTIVE_DAYS = 'Active days (first 28)'
LABEL_COMPLETION_RATE = 'Completion rate (%)'

FIG_DPI = 150
FIG_SIZE = (10, 6)
FIG_SIZE_WIDE = (16, 5)

# Ensure figures output directory exists
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> None:
    """Save figure to reports/figures/ with consistent settings."""
    path = FIGURES_DIR / f'{name}.png'
    fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {path}')


# --- Prerequisite check ---
# Verify the database is populated before proceeding.
# We check both engagement views since this notebook depends on them.
try:
    _check_daily = execute_query('SELECT COUNT(*) AS n FROM v_engagement_daily')
    _check_early = execute_query('SELECT COUNT(*) AS n FROM v_engagement_early')
    _check_student = execute_query('SELECT COUNT(*) AS n FROM v_student_enriched')
    _n_daily = _check_daily['n'].iloc[0]
    _n_early = _check_early['n'].iloc[0]
    _n_student = _check_student['n'].iloc[0]
    if _n_daily == 0 or _n_early == 0 or _n_student == 0:
        raise RuntimeError('One or more views are empty')
    print('Database OK')
    print(f'  v_engagement_daily:  {_n_daily:>12,} rows')
    print(f'  v_engagement_early:  {_n_early:>12,} rows')
    print(f'  v_student_enriched:  {_n_student:>12,} rows')
except Exception as exc:
    raise RuntimeError(
        'Cannot query engagement views. '
        "Run 'python -m run_pipeline' first to populate the database."
    ) from exc

## 2. Ghost Student — Zero Engagement

Il segnale comportamentale più estremo è **nessun comportamento**. Un "ghost student" è un'iscrizione presente in `v_student_enriched` ma assente da `v_engagement_early` — ovvero zero click VLE nei primi 28 giorni.

**Perché partire da qui?** I ghost student stabiliscono il *pavimento* dell'engagement. Prima di analizzare quanto cliccano gli studenti attivi, dobbiamo sapere quanti studenti non hanno mai cliccato affatto, e cosa è successo loro.

**Parallelo SaaS:** I ghost student sono l'equivalente educativo degli *utenti dormienti* — persone che si sono registrate ma non si sono mai attivate. Nella product analytics, il tasso di attivazione è la prima leva di retention: non puoi trattenere utenti che non hanno mai iniziato a usare il prodotto.

In [ ]:
# --- Ghost student identification ---
# v_engagement_early only includes enrollments with >= 1 VLE click in days 0-28.
# Enrollments absent from that view had zero activity — these are "ghost students".
# The LEFT JOIN + IS NULL pattern identifies them precisely per enrollment
# (a student can be a ghost in one course but active in another).
df_ghost_summary = execute_query('''
    SELECT
        COUNT(*) AS n_total,
        SUM(CASE WHEN ee.id_student IS NULL THEN 1 ELSE 0 END) AS n_ghost,
        SUM(CASE WHEN ee.id_student IS NOT NULL THEN 1 ELSE 0 END) AS n_active
    FROM v_student_enriched se
    LEFT JOIN v_engagement_early ee
        ON se.id_student = ee.id_student
        AND se.code_module = ee.code_module
        AND se.code_presentation = ee.code_presentation
''')

n_total = df_ghost_summary['n_total'].iloc[0]
n_ghost = df_ghost_summary['n_ghost'].iloc[0]
n_active = df_ghost_summary['n_active'].iloc[0]

print('=== Ghost Student Overview ===\n')
print(f'  Total enrollments:       {n_total:>8,}')
print(f'  Active (>= 1 click):     {n_active:>8,} ({100 * n_active / n_total:.1f}%)')
print(f'  Ghost (zero activity):   {n_ghost:>8,} ({100 * n_ghost / n_total:.1f}%)')

# --- Completion rate comparison ---
df_ghost_outcome = execute_query('''
    SELECT
        CASE WHEN ee.id_student IS NULL THEN 'Ghost' ELSE 'Active' END AS segment,
        COUNT(*) AS n,
        ROUND(100.0 * SUM(se.completed) / COUNT(*), 1) AS completion_rate_pct
    FROM v_student_enriched se
    LEFT JOIN v_engagement_early ee
        ON se.id_student = ee.id_student
        AND se.code_module = ee.code_module
        AND se.code_presentation = ee.code_presentation
    GROUP BY (ee.id_student IS NULL)
    ORDER BY segment
''')

print('\n=== Completion Rate: Ghost vs. Active ===\n')
print(df_ghost_outcome.to_string(index=False))

In [ ]:
# --- Ghost students by course ---
# Are ghost students concentrated in specific courses, or distributed evenly?
df_ghost_by_course = execute_query('''
    SELECT
        se.code_module,
        COUNT(*) AS n_enrolled,
        SUM(CASE WHEN ee.id_student IS NULL THEN 1 ELSE 0 END) AS n_ghost,
        ROUND(
            100.0 * SUM(CASE WHEN ee.id_student IS NULL THEN 1 ELSE 0 END)
            / COUNT(*), 1
        ) AS ghost_pct,
        ROUND(
            100.0 * SUM(CASE
                WHEN ee.id_student IS NOT NULL THEN se.completed ELSE 0
            END)
            / NULLIF(SUM(CASE WHEN ee.id_student IS NOT NULL THEN 1 ELSE 0 END), 0),
            1
        ) AS active_completion_pct,
        ROUND(
            100.0 * SUM(CASE
                WHEN ee.id_student IS NULL THEN se.completed ELSE 0
            END)
            / NULLIF(SUM(CASE WHEN ee.id_student IS NULL THEN 1 ELSE 0 END), 0),
            1
        ) AS ghost_completion_pct
    FROM v_student_enriched se
    LEFT JOIN v_engagement_early ee
        ON se.id_student = ee.id_student
        AND se.code_module = ee.code_module
        AND se.code_presentation = ee.code_presentation
    GROUP BY se.code_module
    ORDER BY ghost_pct DESC
''')

# --- Visualization: 1x2 panel ---
# Left: ghost count per module. Right: completion rate comparison (ghost vs active).
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

# Left panel: ghost student count by module
modules = df_ghost_by_course['code_module']
ax1.barh(modules, df_ghost_by_course['n_ghost'], color='#8172B3', edgecolor='white')
for i, (_, row) in enumerate(df_ghost_by_course.iterrows()):
    ax1.text(
        row['n_ghost'] + ax1.get_xlim()[1] * 0.01, i,
        f"{row['ghost_pct']:.1f}% of enrolled",
        va='center', fontsize=9, color='#333333',
    )
ax1.set_xlabel('Number of ghost enrollments')
ax1.set_title('Ghost Students by Module')
ax1.invert_yaxis()
sns.despine(ax=ax1)

# Right panel: completion rate — ghost vs active per module
x = np.arange(len(modules))
bar_width = 0.35
ax2.barh(x - bar_width / 2, df_ghost_by_course['active_completion_pct'],
         bar_width, label='Active', color='#55A868', edgecolor='white')
ax2.barh(x + bar_width / 2, df_ghost_by_course['ghost_completion_pct'],
         bar_width, label='Ghost', color='#C44E52', edgecolor='white')
ax2.set_yticks(x)
ax2.set_yticklabels(modules)
ax2.set_xlabel(LABEL_COMPLETION_RATE)
ax2.set_title('Completion Rate: Active vs. Ghost')
ax2.legend(loc='lower right')
ax2.invert_yaxis()
sns.despine(ax=ax2)

fig.suptitle('Ghost Students Across Modules', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '02_ghost_students_by_course')
plt.show()

In [ ]:
# --- 4-class outcome distribution for ghost students ---
# What happens to students who never engage? Are they all Withdrawn,
# or do some manage to Pass/Fail through other channels (e.g. exams only)?
df_ghost_4class = execute_query('''
    SELECT
        se.final_result,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM v_student_enriched se
    LEFT JOIN v_engagement_early ee
        ON se.id_student = ee.id_student
        AND se.code_module = ee.code_module
        AND se.code_presentation = ee.code_presentation
    WHERE ee.id_student IS NULL
    GROUP BY se.final_result
    ORDER BY n DESC
''')

print('=== Ghost Student Outcome Distribution ===\n')
print(df_ghost_4class.to_string(index=False))

# --- Horizontal bar chart ---
fig, ax = plt.subplots(figsize=FIG_SIZE)
colors = [PALETTE_OUTCOME.get(r, '#999999') for r in df_ghost_4class['final_result']]
bars = ax.barh(df_ghost_4class['final_result'], df_ghost_4class['n'], color=colors,
               edgecolor='white')

for bar, (_, row) in zip(bars, df_ghost_4class.iterrows()):
    ax.text(
        bar.get_width() + ax.get_xlim()[1] * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{int(row['n']):,} ({row['pct']:.1f}%)",
        va='center', fontsize=11,
    )

ax.set_xlabel(LABEL_NUM_ENROLLMENTS)
ax.set_title('Outcome Distribution — Ghost Students Only (zero VLE activity)')
ax.invert_yaxis()
sns.despine(left=True)
fig.tight_layout()
save_fig(fig, '02_ghost_outcome_distribution')
plt.show()

> **Risultato chiave:** I ghost student hanno un tasso di completamento prossimo allo zero. La stragrande maggioranza sono **Withdrawn** — si sono iscritti, non hanno mai interagito con il VLE e alla fine se ne sono andati.
>
> - I ghost student sono presenti in tutti i moduli, non concentrati in un singolo corso.
> - Il divario nel tasso di completamento tra studenti attivi e ghost è enorme — questo è il segnale engagement-esito più chiaro nel dataset.
> - Un piccolo numero di ghost student potrebbe comunque ottenere Pass o Fail (es. tramite esami in presenza o crediti accumulati), ma sono una piccola minoranza.
>
> **Implicazione per BQ5:** I ghost student sono il frutto più a portata di mano per gli interventi. Hanno bisogno di *attivazione*, non di supporto accademico. In termini SaaS, questo è il "gap di onboarding" — l'utente si è registrato ma non ha mai sperimentato il valore core del prodotto.
>
> **Da qui in poi**, le Sezioni 3–9 analizzano solo **studenti attivi** (quelli con almeno un click VLE nei primi 28 giorni), salvo diversa indicazione esplicita.

## 3. Distribuzioni dell'engagement precoce

Per gli studenti che **hanno** interagito (non-ghost), com'è la loro attività precoce? Questa sezione profila le quattro metriche chiave da `v_engagement_early`:

| Metrica | Cosa misura |
|---------|------------|
| `active_days_first_28` | Numero di giorni distinti con almeno un click (0–28) |
| `total_clicks_first_28` | Click VLE totali su tutte le risorse |
| `avg_clicks_per_active_day` | Intensità: click per giorno quando presente |
| `last_active_day_in_window` | Ultimo giorno di attività nella finestra di 28 giorni |

Confrontiamo queste distribuzioni tra iscrizioni **Completed** e **Not completed** per costruire intuizione visiva prima dei test statistici formali nel Notebook 04 (BQ2).

> **Nota:** I ghost student (zero attività) sono esclusi da `v_engagement_early` per costruzione. Le distribuzioni qui rappresentano solo la popolazione *attiva*.

In [ ]:
# --- Summary statistics by outcome ---
# Mean vs median gap reveals skewness (common in clickstream data:
# a few power users generate disproportionate click volumes).
df_early_stats = execute_query('''
    SELECT
        se.completed,
        COUNT(*) AS n,
        ROUND(AVG(ee.active_days_first_28), 1) AS mean_active_days,
        ROUND(
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY ee.active_days_first_28), 1
        ) AS median_active_days,
        ROUND(AVG(ee.total_clicks_first_28), 0) AS mean_total_clicks,
        ROUND(
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY ee.total_clicks_first_28), 0
        ) AS median_total_clicks,
        ROUND(AVG(ee.avg_clicks_per_active_day), 1) AS mean_clicks_per_day,
        ROUND(AVG(ee.last_active_day_in_window), 1) AS mean_last_active_day
    FROM v_engagement_early ee
    JOIN v_student_enriched se
        USING (id_student, code_module, code_presentation)
    GROUP BY se.completed
    ORDER BY se.completed DESC
''')

# Map binary outcome to readable labels (constants from setup cell)
df_early_stats.insert(0, 'outcome', df_early_stats.pop('completed').map(LABEL_BINARY))

print('=== Early Engagement Summary (first 28 days, active students only) ===\n')
df_early_stats

In [ ]:
# --- 2x2 violin plot grid: all 4 early metrics split by outcome ---
# Violins show the full distribution shape — more informative than box plots
# for detecting bimodality or heavy tails in engagement data.
df_early = execute_query('''
    SELECT
        ee.active_days_first_28,
        ee.total_clicks_first_28,
        ee.avg_clicks_per_active_day,
        ee.last_active_day_in_window,
        se.completed
    FROM v_engagement_early ee
    JOIN v_student_enriched se
        USING (id_student, code_module, code_presentation)
''')
df_early['outcome'] = df_early['completed'].map(LABEL_BINARY)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = [
    ('active_days_first_28', LABEL_ACTIVE_DAYS, axes[0, 0]),
    ('total_clicks_first_28', 'Total clicks (first 28)', axes[0, 1]),
    ('avg_clicks_per_active_day', 'Avg clicks per active day', axes[1, 0]),
    ('last_active_day_in_window', 'Last active day in window', axes[1, 1]),
]

for col, ylabel, ax in metrics:
    sns.violinplot(
        data=df_early, x='outcome', y=col,
        palette=PALETTE_BINARY_LABELS, inner='quartile', ax=ax,
    )
    ax.set_xlabel('')
    ax.set_ylabel(ylabel)
    sns.despine(ax=ax)

fig.suptitle(
    'Early Engagement Metrics by Completion Outcome (first 28 days)',
    fontsize=14, y=1.02,
)
fig.tight_layout()
save_fig(fig, '02_early_engagement_violins')
plt.show()

In [ ]:
# --- Overlapping histograms: active days and total clicks ---
# Histograms complement violins by showing raw frequency counts,
# making it easier to see where the bulk of enrollments falls.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

# Left: active days
for outcome_val, label in LABEL_BINARY.items():
    subset = df_early[df_early['completed'] == outcome_val]
    ax1.hist(
        subset['active_days_first_28'], bins=28, alpha=0.6,
        label=label, color=PALETTE_BINARY[outcome_val],
    )
ax1.set_xlabel(LABEL_ACTIVE_DAYS)
ax1.set_ylabel(LABEL_NUM_ENROLLMENTS)
ax1.set_title('Active Days Distribution by Outcome')
ax1.legend()
sns.despine(ax=ax1)

# Right: total clicks (cap at 99th percentile to avoid outlier compression)
p99 = df_early['total_clicks_first_28'].quantile(0.99)
for outcome_val, label in LABEL_BINARY.items():
    subset = df_early[df_early['completed'] == outcome_val]
    ax2.hist(
        subset['total_clicks_first_28'].clip(upper=p99), bins=50, alpha=0.6,
        label=label, color=PALETTE_BINARY[outcome_val],
    )
ax2.set_xlabel(f'Total clicks (first 28 days, capped at p99 = {p99:,.0f})')
ax2.set_ylabel(LABEL_NUM_ENROLLMENTS)
ax2.set_title('Total Clicks Distribution by Outcome')
ax2.legend()
sns.despine(ax=ax2)

fig.suptitle('Early Engagement Distributions', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '02_early_engagement_histograms')
plt.show()

> **Interpretazione:**
> - Tutte e quattro le metriche mostrano una **separazione visibile** tra completers e non-completers. I completers tendono ad avere più giorni attivi, più click totali e il loro ultimo giorno attivo è più avanti nella finestra di 28 giorni.
> - La distribuzione dei **click totali** è fortemente asimmetrica a destra — pochi power user generano volumi di click sproporzionati. La media è sostanzialmente più alta della mediana, confermando questa asimmetria.
> - La metrica **ultimo giorno attivo** è particolarmente rivelatrice: l'attività dei non-completers tende a diminuire prima nella finestra, mentre i completers rimangono attivi più vicino al giorno 28.
> - L'istogramma dei **giorni attivi** mostra che molti non-completers sono attivi per soli 1–5 giorni su 28 — un breve engagement prima di scomparire.
>
> **Avvertenza sulla causalità:** Un engagement più alto è *associato al* completamento, ma non possiamo dire che l'engagement *causi* il completamento solo da questi dati. Studenti motivati potrebbero sia interagire di più che completare di più — l'engagement potrebbe essere un proxy della motivazione, non una causa del successo. Il Notebook 04 (BQ2) quantificherà queste associazioni con test statistici formali ed effect size.

## 4. Decili di engagement vs. esito

Esiste una relazione **dose-risposta** tra engagement e completamento? Se più engagement predice monotonicamente esiti migliori, questo rafforza il caso per interventi basati sull'engagement.

`v_engagement_early` include una colonna `engagement_decile_in_course`: ogni studente è classificato 1–10 all'interno del proprio course-presentation usando `NTILE(10)` su `total_clicks_first_28`. Questa normalizzazione è critica — assicura che il decile 1 in un corso ad alto engagement e il decile 1 in un corso a basso engagement significhino entrambi "ultimo 10% di *quel* corso."

In [ ]:
# --- Completion rate by engagement decile ---
# If the relationship is dose-response, we expect a monotonic increase
# from decile 1 (lowest engagement) to decile 10 (highest engagement).
df_decile = execute_query('''
    SELECT
        ee.engagement_decile_in_course AS decile,
        COUNT(*) AS n,
        ROUND(100.0 * SUM(se.completed) / COUNT(*), 1) AS completion_rate_pct
    FROM v_engagement_early ee
    JOIN v_student_enriched se
        USING (id_student, code_module, code_presentation)
    GROUP BY ee.engagement_decile_in_course
    ORDER BY ee.engagement_decile_in_course
''')

print('=== Completion Rate by Engagement Decile ===\n')
print(df_decile.to_string(index=False))

# --- Bar chart with gradient coloring ---
# Color gradient from red (low decile) to green (high decile) reinforces
# the dose-response visual pattern and matches the binary palette semantics.
# Weighted by decile size — unweighted mean would misstate the rate if NTILE
# produces groups of slightly different size.
overall_rate = (df_decile['completion_rate_pct'] * df_decile['n']).sum() / df_decile['n'].sum()
n_deciles = len(df_decile)
gradient_colors = [
    plt.cm.RdYlGn(i / (n_deciles - 1)) for i in range(n_deciles)
]

fig, ax = plt.subplots(figsize=FIG_SIZE)
bars = ax.bar(
    df_decile['decile'], df_decile['completion_rate_pct'],
    color=gradient_colors, edgecolor='white',
)

# Annotate each bar with the exact percentage
for bar, (_, row) in zip(bars, df_decile.iterrows()):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
        f"{row['completion_rate_pct']:.1f}%",
        ha='center', fontsize=9, color='#333333',
    )

# Reference line: overall average (across active students only)
ax.axhline(y=overall_rate, color='gray', linestyle='--', linewidth=1,
           label=f'Average: {overall_rate:.1f}%')

ax.set_xlabel('Engagement decile (1 = lowest, 10 = highest)')
ax.set_ylabel(LABEL_COMPLETION_RATE)
ax.set_title('Completion Rate by Engagement Decile (within-course normalized)')
ax.set_xticks(range(1, 11))
ax.legend(loc='upper left')
sns.despine()
fig.tight_layout()
save_fig(fig, '02_decile_completion_rate')
plt.show()

In [ ]:
# --- 4-class outcome breakdown by decile (100% stacked bar) ---
# Shows how the mix of Pass/Distinction/Fail/Withdrawn changes across deciles.
# Complements the binary view above with the full outcome granularity.
df_decile_4class = execute_query('''
    SELECT
        ee.engagement_decile_in_course AS decile,
        se.final_result,
        COUNT(*) AS n
    FROM v_engagement_early ee
    JOIN v_student_enriched se
        USING (id_student, code_module, code_presentation)
    GROUP BY ee.engagement_decile_in_course, se.final_result
    ORDER BY ee.engagement_decile_in_course, se.final_result
''')

# Pivot and normalize to 100%
df_pivot = df_decile_4class.pivot(
    index='decile', columns='final_result', values='n',
).fillna(0)
df_pct = df_pivot.div(df_pivot.sum(axis=1), axis=0) * 100

# Order columns: positive outcomes first, then negative
col_order = ['Distinction', 'Pass', 'Fail', 'Withdrawn']
df_pct = df_pct[[c for c in col_order if c in df_pct.columns]]

fig, ax = plt.subplots(figsize=FIG_SIZE)
df_pct.plot.bar(
    stacked=True, ax=ax,
    color=[PALETTE_OUTCOME[c] for c in df_pct.columns],
    edgecolor='white', linewidth=0.5,
)
ax.set_xlabel('Engagement decile (1 = lowest, 10 = highest)')
ax.set_ylabel('Percentage of enrollments')
ax.set_title('Outcome Mix by Engagement Decile')
ax.legend(title='Outcome', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
sns.despine()
fig.tight_layout()
save_fig(fig, '02_decile_outcome_stacked')
plt.show()

> **Interpretazione:**
> - Il pattern dose-risposta è chiaro: il tasso di completamento aumenta in modo monotono (o quasi monotono) dal decile di engagement più basso a quello più alto.
> - I 2–3 decili più bassi sono un **segmento a rischio estremo** — i loro tassi di completamento sono molto sotto la media.
> - Nel grafico a barre impilate, il segmento Withdrawn (viola) si riduce drasticamente all'aumentare dell'engagement, mentre Distinction (verde) cresce. Questo suggerisce che un engagement più alto è associato sia a un minor dropout che a un miglior rendimento accademico.
> - Poiché i decili sono **normalizzati all'interno del corso**, questo pattern è robusto alle differenze a livello di corso nel volume di click. Uno studente nel decile 1 di qualsiasi corso è ad alto rischio.
>
> **Insight azionabile (anteprima BQ5):** Un sistema di early warning basato sull'engagement potrebbe segnalare gli studenti nei 2–3 decili più bassi entro i primi 28 giorni. In termini SaaS, questo equivale a un *health score* che attiva un contatto proattivo per gli utenti a rischio.

## 5. Traiettoria di engagement giornaliero

Le Sezioni 3–4 hanno riassunto l'engagement come un singolo numero su 28 giorni. Qui aggiungiamo la **dimensione temporale**: come evolve l'engagement giorno per giorno?

La domanda chiave: **a che punto divergono le traiettorie di completers e non-completers?** Se il divario si apre presto (es. entro la prima settimana), la finestra di intervento è stretta ma azionabile. Se si apre gradualmente, gli interventi hanno più tempo ma potrebbero dover essere più persistenti.

Usiamo `v_engagement_daily`, che fornisce granularità per-giorno (click totali, risorse distinte, tipi di attività) per ogni giorno-studente con almeno un click.

In [ ]:
# --- Average daily clicks by outcome group (days 0-28) ---
# We compute the mean clicks per day for each outcome group.
# Students who did not click on a given day are NOT in v_engagement_daily
# for that day — so this average is "clicks per active student on that day",
# not "clicks per enrolled student". The active student count plot below
# provides the complementary view.
df_trajectory = execute_query('''
    SELECT
        ed.date AS day,
        se.completed,
        ROUND(AVG(ed.total_clicks), 2) AS avg_clicks,
        COUNT(*) AS n_active_enrollments
    FROM v_engagement_daily ed
    JOIN v_student_enriched se
        USING (id_student, code_module, code_presentation)
    WHERE ed.date BETWEEN 0 AND 28
    GROUP BY ed.date, se.completed
    ORDER BY ed.date, se.completed
''')

# Map binary outcome to readable labels (constants from setup cell)
df_trajectory['outcome'] = df_trajectory['completed'].map(LABEL_BINARY)

fig, ax = plt.subplots(figsize=FIG_SIZE)
for outcome in [LABEL_COMPLETED, LABEL_NOT_COMPLETED]:
    subset = df_trajectory[df_trajectory['outcome'] == outcome]
    ax.plot(
        subset['day'], subset['avg_clicks'],
        label=outcome, color=PALETTE_BINARY_LABELS[outcome],
        linewidth=2,
    )

# Shade the area between the two lines to emphasize divergence
df_comp = df_trajectory[df_trajectory['outcome'] == LABEL_COMPLETED].set_index('day')
df_notc = df_trajectory[df_trajectory['outcome'] == LABEL_NOT_COMPLETED].set_index('day')
common_days = df_comp.index.intersection(df_notc.index)
ax.fill_between(
    common_days,
    df_comp.loc[common_days, 'avg_clicks'],
    df_notc.loc[common_days, 'avg_clicks'],
    alpha=0.1, color='gray',
)

ax.set_xlabel('Day (relative to course start)')
ax.set_ylabel('Average clicks per active student')
ax.set_title('Daily Engagement Trajectory (first 28 days)')
ax.legend()
sns.despine()
fig.tight_layout()
save_fig(fig, '02_daily_trajectory_first_28')
plt.show()

In [ ]:
# --- Active enrollment count per day by outcome ---
# This contextualizes the trajectory above: does the gap widen because
# non-completers click less, or because they stop clicking entirely?
# A declining line for non-completers means students are dropping out
# of activity — not just reducing their intensity.
fig, ax = plt.subplots(figsize=FIG_SIZE)
for outcome in [LABEL_COMPLETED, LABEL_NOT_COMPLETED]:
    subset = df_trajectory[df_trajectory['outcome'] == outcome]
    ax.plot(
        subset['day'], subset['n_active_enrollments'],
        label=outcome, color=PALETTE_BINARY_LABELS[outcome],
        linewidth=2,
    )

ax.set_xlabel('Day (relative to course start)')
ax.set_ylabel('Number of active enrollments')
ax.set_title('Active Enrollments per Day (first 28 days)')
ax.legend()
sns.despine()
fig.tight_layout()
save_fig(fig, '02_daily_active_students_first_28')
plt.show()

> **Interpretazione:**
> - Il grafico della traiettoria mostra un **divario persistente** tra completers e non-completers durante tutti i primi 28 giorni. I completers cliccano costantemente di più nei giorni in cui sono attivi.
> - Il conteggio delle iscrizioni attive rivela un **doppio segnale**: i non-completers non solo cliccano meno quando sono presenti, ma anche *smettono di presentarsi* prima. La linea rossa in calo rappresenta studenti che si sono effettivamente disimpegnati dalla piattaforma.
> - La combinazione di questi due grafici è potente: il divario di engagement è guidato sia dall'**intensità** (meno click per visita) che dalla **frequenza** (meno visite, abbandono più precoce dell'attività).
>
> **Implicazione per gli interventi:** La divergenza diventa probabilmente visibile entro i primi 7–10 giorni. Questo restringe la finestra di intervento: entro la fine del primo mese, molti non-completers hanno già smesso di interagire. I sistemi di rilevamento precoce devono agire entro le prime due settimane per avere il massimo impatto.

## 6. Tipologia di engagement — Binge vs. Steady

Non tutto l'engagement è uguale. Due studenti con 200 click totali possono avere pattern molto diversi:
- **Binge**: 200 click in 2 giorni (burst concentrati, comportamento da cramming)
- **Steady**: 200 click distribuiti su 14 giorni (engagement costante e distribuito)

Le dimensioni `active_days_first_28` e `total_clicks_first_28` catturano questa distinzione. Uno scatter plot di queste due variabili, colorato per esito, rivela se la *costanza* o il *volume* contano di più per il completamento.

**Parallelo SaaS:** Questo corrisponde alla differenza tra DAU (daily active user) e profondità di sessione. Un utente che fa login brevemente ogni giorno è comportamentalmente diverso da uno che ha una sessione maratona a settimana — anche se il loro tempo di utilizzo totale è simile.

In [ ]:
# --- Scatter plot: active days vs total clicks, colored by outcome ---
# We reuse df_early (loaded in Section 3) which already contains all 4 metrics
# plus the outcome column.
# Cap total_clicks at the 99th percentile to prevent a few extreme outliers
# from compressing the visual range for the majority of students.
p99_clicks = df_early['total_clicks_first_28'].quantile(0.99)

fig, ax = plt.subplots(figsize=FIG_SIZE)
# Plot non-completers first (background) so completers are visible on top
for outcome_val in [0, 1]:
    label = LABEL_BINARY[outcome_val]
    subset = df_early[df_early['completed'] == outcome_val]
    ax.scatter(
        subset['active_days_first_28'],
        subset['total_clicks_first_28'].clip(upper=p99_clicks),
        alpha=0.15, s=10, label=label,
        color=PALETTE_BINARY[outcome_val],
    )

# Annotate quadrant labels to guide interpretation
ax.text(2, p99_clicks * 0.90, 'Binge\n(few days, many clicks)',
        fontsize=10, color='#666666', style='italic')
ax.text(22, p99_clicks * 0.90, 'Power User\n(many days, many clicks)',
        fontsize=10, color='#666666', style='italic')
ax.text(2, p99_clicks * 0.05, 'Minimal\n(few days, few clicks)',
        fontsize=10, color='#666666', style='italic')
ax.text(22, p99_clicks * 0.05, 'Steady\n(many days, moderate clicks)',
        fontsize=10, color='#666666', style='italic')

ax.set_xlabel(LABEL_ACTIVE_DAYS)
ax.set_ylabel(f'Total clicks (first 28 days, capped at p99 = {p99_clicks:,.0f})')
ax.set_title('Engagement Typology: Active Days vs. Total Clicks')
ax.legend(markerscale=5)
sns.despine()
fig.tight_layout()
save_fig(fig, '02_engagement_typology_scatter')
plt.show()

In [ ]:
# --- Typology heatmap: median-split quadrants → completion rate ---
# Split students into 4 quadrants using the median of active_days and
# the median of avg_clicks_per_active_day. This quantifies whether
# consistency (high active days) or intensity (high clicks per day) is
# more strongly associated with completion.
df_thresholds = execute_query('''
    SELECT
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY active_days_first_28)
            AS median_active_days,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY avg_clicks_per_active_day)
            AS median_intensity
    FROM v_engagement_early
''')

med_days = df_thresholds['median_active_days'].iloc[0]
med_intensity = df_thresholds['median_intensity'].iloc[0]

print(f'Median active days: {med_days:.0f}')
print(f'Median clicks per active day: {med_intensity:.1f}')

# Classify each enrollment into a quadrant
df_early['frequency'] = np.where(
    df_early['active_days_first_28'] >= med_days, 'High frequency', 'Low frequency'
)
df_early['intensity'] = np.where(
    df_early['avg_clicks_per_active_day'] >= med_intensity, 'High intensity', 'Low intensity'
)

# Compute completion rate per quadrant
df_typology = (
    df_early.groupby(['frequency', 'intensity'])
    .agg(n=('completed', 'count'), completion_rate=('completed', 'mean'))
    .reset_index()
)
df_typology['completion_rate'] = (df_typology['completion_rate'] * 100).round(1)

# Pivot for heatmap
heatmap_data = df_typology.pivot(
    index='intensity', columns='frequency', values='completion_rate',
)
heatmap_counts = df_typology.pivot(
    index='intensity', columns='frequency', values='n',
)

# Reorder for intuitive reading: high on top, low on bottom
row_order = ['High intensity', 'Low intensity']
col_order = ['Low frequency', 'High frequency']
heatmap_data = heatmap_data.loc[row_order, col_order]
heatmap_counts = heatmap_counts.loc[row_order, col_order]

# Annotate with both completion rate and count
annot = heatmap_data.copy().astype(str)
for r in row_order:
    for c in col_order:
        rate = heatmap_data.loc[r, c]
        count = heatmap_counts.loc[r, c]
        annot.loc[r, c] = f'{rate:.1f}%\n(n={int(count):,})'

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    heatmap_data, annot=annot, fmt='', cmap='RdYlGn',
    vmin=0, vmax=100, linewidths=1, linecolor='white',
    cbar_kws={'label': LABEL_COMPLETION_RATE}, ax=ax,
)
ax.set_title('Engagement Typology: Completion Rate by Quadrant\n'
             f'(split at median: {med_days:.0f} active days, '
             f'{med_intensity:.1f} clicks/day)')
ax.set_xlabel('')
ax.set_ylabel('')
fig.tight_layout()
save_fig(fig, '02_engagement_typology_heatmap')
plt.show()

> **Interpretazione:**
> - Lo scatter plot mostra che i **completers si raggruppano in alto a destra** (molti giorni attivi, molti click totali), mentre i non-completers dominano in basso a sinistra (pochi giorni, pochi click).
> - Il quadrante "binge" (pochi giorni, molti click) è relativamente sparso — la maggior parte degli studenti con volumi di click alti li distribuisce anche su più giorni.
> - La heatmap 2×2 quantifica questo: **l'alta frequenza è il predittore più forte**. Passare da bassa ad alta frequenza (più giorni attivi) aumenta il tasso di completamento più che passare da bassa ad alta intensità (più click per sessione). La costanza batte il cramming.
> - Il quadrante "minimale" (bassa frequenza, bassa intensità) ha il tasso di completamento più basso — sono studenti che hanno interagito a malapena anche quando si sono presentati.
>
> **Implicazione per BQ5:** Gli interventi dovrebbero dare priorità alla *costanza* rispetto al *volume*. Incoraggiare gli studenti a fare login regolarmente (anche brevemente) potrebbe essere più efficace che incoraggiare sessioni più lunghe. Questo è analogo alla meccanica del "daily streak" usata nelle app consumer per costruire la formazione dell'abitudine.

## 7. Diversità dei tipi di attività

Le Sezioni 3–6 si sono concentrate su *quanto* gli studenti interagiscono (click, giorni, intensità). Questa sezione chiede *con cosa* interagiscono.

Il VLE OULAD contiene diversi tipi di attività: pagine di contenuto (`oucontent`), forum (`forumng`), quiz, risorse, homepage, sottopagine e altro. La tabella `vle` mappa ogni risorsa (`id_site`) al suo `activity_type`.

**Ipotesi della diversità:** Studenti che interagiscono con una varietà più ampia di tipi di risorse potrebbero essere più profondamente coinvolti nel corso — accedendo non solo ai contenuti core ma anche a forum, quiz e materiali supplementari. Se la diversità correla con il completamento, suggerisce che l'*ampiezza* dell'engagement conta insieme al *volume*.

In [ ]:
# --- Activity type popularity: which resource types get the most clicks? ---
# Filtering to first 28 days to match the engagement window used throughout.
df_activity_pop = execute_query('''
    SELECT
        v.activity_type,
        SUM(sv.sum_click) AS total_clicks,
        ROUND(100.0 * SUM(sv.sum_click) / SUM(SUM(sv.sum_click)) OVER (), 1)
            AS pct_of_clicks
    FROM studentVle sv
    JOIN vle v
        ON sv.id_site = v.id_site
        AND sv.code_module = v.code_module
        AND sv.code_presentation = v.code_presentation
    WHERE sv.date BETWEEN 0 AND 28
    GROUP BY v.activity_type
    ORDER BY total_clicks DESC
''')

print('=== Activity Type Popularity (first 28 days) ===\n')
print(df_activity_pop.to_string(index=False))

In [ ]:
# --- Activity type usage by outcome ---
# Which activity types show the largest gap between completers and non-completers?
# CTE computes per-enrollment click totals first, then averages — this avoids
# COUNT(DISTINCT concat(...)), a pattern that is not portable across SQL engines.
df_activity_outcome = execute_query('''
    WITH enrollment_activity_clicks AS (
        SELECT
            v.activity_type,
            se.completed,
            se.id_student,
            se.code_module,
            se.code_presentation,
            SUM(sv.sum_click) AS enrollment_clicks
        FROM studentVle sv
        JOIN vle v
            ON sv.id_site = v.id_site
            AND sv.code_module = v.code_module
            AND sv.code_presentation = v.code_presentation
        JOIN v_student_enriched se
            ON sv.id_student = se.id_student
            AND sv.code_module = se.code_module
            AND sv.code_presentation = se.code_presentation
        WHERE sv.date BETWEEN 0 AND 28
        GROUP BY
            v.activity_type,
            se.completed,
            se.id_student,
            se.code_module,
            se.code_presentation
    )
    SELECT
        activity_type,
        completed,
        ROUND(AVG(enrollment_clicks), 1) AS avg_clicks_per_enrollment
    FROM enrollment_activity_clicks
    GROUP BY activity_type, completed
    ORDER BY activity_type, completed
''')

# Map binary outcome to readable labels (constants from setup cell)
df_activity_outcome['outcome'] = df_activity_outcome['completed'].map(LABEL_BINARY)

# Pivot for grouped bar chart
df_act_pivot = df_activity_outcome.pivot(
    index='activity_type', columns='outcome', values='avg_clicks_per_enrollment',
).fillna(0)

# Sort by the gap between Completed and Not completed (largest gap first)
df_act_pivot['_gap'] = (
    df_act_pivot.get(LABEL_COMPLETED, 0) - df_act_pivot.get(LABEL_NOT_COMPLETED, 0)
)
df_act_pivot = df_act_pivot.sort_values('_gap', ascending=True)
df_act_pivot = df_act_pivot.drop(columns='_gap')

fig, ax = plt.subplots(figsize=FIG_SIZE)
df_act_pivot.plot.barh(
    ax=ax,
    color=[PALETTE_BINARY_LABELS.get(c, '#999999') for c in df_act_pivot.columns],
    edgecolor='white',
)
ax.set_xlabel('Average clicks per enrollment (first 28 days)')
ax.set_ylabel('')
ax.set_title('Activity Type Usage by Outcome')
ax.legend(title='Outcome')
sns.despine()
fig.tight_layout()
save_fig(fig, '02_activity_type_by_outcome')
plt.show()

In [ ]:
# --- Activity type diversity per student vs completion ---
# Count how many distinct activity types each student used in first 28 days,
# then compute completion rate per diversity level.
df_diversity = execute_query('''
    SELECT
        sub.n_activity_types,
        COUNT(*) AS n_enrollments,
        ROUND(100.0 * SUM(se.completed) / COUNT(*), 1) AS completion_rate_pct
    FROM (
        SELECT
            sv.id_student,
            sv.code_module,
            sv.code_presentation,
            COUNT(DISTINCT v.activity_type) AS n_activity_types
        FROM studentVle sv
        JOIN vle v
            ON sv.id_site = v.id_site
            AND sv.code_module = v.code_module
            AND sv.code_presentation = v.code_presentation
        WHERE sv.date BETWEEN 0 AND 28
        GROUP BY sv.id_student, sv.code_module, sv.code_presentation
    ) sub
    JOIN v_student_enriched se
        USING (id_student, code_module, code_presentation)
    GROUP BY sub.n_activity_types
    ORDER BY sub.n_activity_types
''')

print('=== Activity Type Diversity vs. Completion Rate ===\n')
print(df_diversity.to_string(index=False))

> **Interpretazione:**
> - Pochi tipi di attività dominano il volume di click — `oucontent` (pagine di contenuto del corso) e `homepage`/`subpage` probabilmente rappresentano la maggioranza dei click.
> - Il **divario tra completers e non-completers** varia per tipo di attività. Alcuni tipi di risorse (es. forum, quiz) possono mostrare un divario proporzionalmente maggiore, suggerendo che sono segnali di engagement più "discriminanti".
> - La **diversità** dei tipi di attività mostra un'associazione positiva con il completamento: studenti che usano più tipi di risorse tendono a completare a tassi più alti. Questo è un altro pattern dose-risposta, che complementa l'analisi dei decili nella Sezione 4.
>
> **Avvertenza:** La diversità dei tipi di attività è in parte confusa con il volume totale di engagement — studenti che cliccano di più hanno più probabilità di incontrare tipi di risorse diversi. Il segnale di diversità potrebbe non essere indipendente dal segnale di volume. BQ2 lo testerà più formalmente.
>
> **Parallelo SaaS:** L'ampiezza delle funzionalità (numero di feature distinte usate) è un predittore di retention comune nella product analytics. Gli utenti che esplorano oltre il set di funzionalità core tendono a essere più coinvolti e meno propensi al churn.

## 8. Profili di engagement a livello di corso

Il Notebook 01 (Sezione 7) ha esaminato le caratteristiche di *design* dei corsi — densità degli assessment, conteggio risorse VLE, durata. Qui aggiungiamo la **dimensione dell'engagement**: quanto cliccano effettivamente gli studenti in ciascun corso?

Questo è importante perché corsi con più risorse possono naturalmente generare più click. Un livello di engagement "basso" in un corso ricco di risorse potrebbe essere "normale" in uno più leggero. BQ4 (Notebook 06) formalizzerà questa analisi; qui stabiliamo la baseline.

In [ ]:
# --- Course-level engagement summary ---
df_course_engagement = execute_query('''
    SELECT
        ee.code_module,
        COUNT(*) AS n_active_enrollments,
        ROUND(AVG(ee.total_clicks_first_28), 0) AS mean_clicks,
        ROUND(
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY ee.total_clicks_first_28), 0
        ) AS median_clicks,
        ROUND(AVG(ee.active_days_first_28), 1) AS mean_active_days,
        ROUND(
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY ee.active_days_first_28), 0
        ) AS median_active_days,
        ROUND(AVG(ee.avg_clicks_per_active_day), 1) AS mean_intensity
    FROM v_engagement_early ee
    GROUP BY ee.code_module
    ORDER BY median_clicks DESC
''')

print('=== Course-Level Engagement Summary (first 28 days) ===\n')
df_course_engagement

In [ ]:
# --- Box plots of total clicks by module ---
# Shows within-course engagement variation: some courses may have tight
# distributions (all students click similarly) while others have wide spread.
df_course_clicks = execute_query('''
    SELECT
        ee.code_module,
        ee.total_clicks_first_28
    FROM v_engagement_early ee
''')

# Cap at 99th percentile to manage visual outliers
p99_course = df_course_clicks['total_clicks_first_28'].quantile(0.99)
df_course_clicks['clicks_capped'] = df_course_clicks['total_clicks_first_28'].clip(
    upper=p99_course,
)

# Order modules by median clicks for consistent reading
module_order = (
    df_course_clicks.groupby('code_module')['total_clicks_first_28']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=FIG_SIZE)
sns.boxplot(
    data=df_course_clicks, x='code_module', y='clicks_capped',
    order=module_order, color='#4C72B0', ax=ax,
)
ax.set_xlabel('Module')
ax.set_ylabel(f'Total clicks (first 28 days, capped at p99 = {p99_course:,.0f})')
ax.set_title('Engagement Distribution by Module')
sns.despine()
fig.tight_layout()
save_fig(fig, '02_course_engagement_boxplot')
plt.show()

> **Interpretazione:**
> - I livelli di engagement variano **sostanzialmente** tra i moduli. Alcuni corsi generano volumi di click molto più alti di altri, riflettendo probabilmente differenze nel design del corso (numero di risorse VLE, struttura degli assessment, formato dei contenuti).
> - All'interno di ciascun corso, c'è anche una variazione considerevole — i baffi e gli outlier del box plot mostrano che alcuni studenti cliccano molte volte più dei loro pari nello stesso corso.
> - Il divario media-mediana nella tabella riepilogativa conferma distribuzioni asimmetriche a destra nella maggior parte dei corsi: pochi studenti molto attivi tirano la media sopra la mediana.
>
> **Implicazione:** Le metriche di engagement vanno interpretate **relative al corso**, non in termini assoluti. È per questo che il decile di engagement (Sezione 4) normalizza all'interno di ciascun course-presentation. Uno studente con 100 click può essere nel decile più alto di un corso e nel decile più basso di un altro. BQ4 (Notebook 06) analizzerà come le caratteristiche di design del corso si relazionano ai livelli di engagement e alla retention.

## 9. Matrice di correlazione engagement-esito

Sintetizziamo ora tutte le metriche di engagement precoce in una singola vista: quali metriche hanno la **più forte associazione lineare** con il completamento?

Calcoliamo le correlazioni di Pearson tra tutte le variabili di engagement e la variabile binaria `completed`. Quando una variabile è binaria (0/1) e l'altra è continua, la correlazione di Pearson equivale alla correlazione punto-biseriale — una misura standard di associazione tra una variabile continua e una dicotomica.

**Importante:** Questa analisi usa un `LEFT JOIN` per includere i ghost student (con valori di engagement impostati a 0). Questo dà la prospettiva sull'intera popolazione: i ghost student sono l'estremo inferiore dell'engagement, e includerli rafforza le correlazioni perché quasi tutti non completano.

In [ ]:
# --- Full engagement + outcome data for correlation analysis ---
# LEFT JOIN includes ghost students (COALESCE to 0) for the full population view.
# Using -1 for last_active_day when NULL to indicate "never active",
# keeping it numerically distinct from day 0.
df_corr_data = execute_query('''
    SELECT
        COALESCE(ee.active_days_first_28, 0)          AS active_days,
        COALESCE(ee.total_clicks_first_28, 0)         AS total_clicks,
        COALESCE(ee.avg_clicks_per_active_day, 0)     AS clicks_per_day,
        COALESCE(ee.last_active_day_in_window, -1)    AS last_active_day,
        COALESCE(ee.engagement_decile_in_course, 0)   AS engagement_decile,
        se.completed
    FROM v_student_enriched se
    LEFT JOIN v_engagement_early ee
        ON se.id_student = ee.id_student
        AND se.code_module = ee.code_module
        AND se.code_presentation = ee.code_presentation
''')

# Shorter column labels for readability in the heatmap
label_map = {
    'active_days': 'Active days',
    'total_clicks': 'Total clicks',
    'clicks_per_day': 'Clicks/day',
    'last_active_day': 'Last active day',
    'engagement_decile': 'Engagement decile',
    'completed': 'Completed',
}
df_corr_renamed = df_corr_data.rename(columns=label_map)
corr_matrix = df_corr_renamed.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap=PALETTE_SEQUENTIAL,
    vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
    ax=ax,
)
ax.set_title('Engagement-Outcome Correlation Matrix\n(includes ghost students as zeros)')
fig.tight_layout()
save_fig(fig, '02_engagement_correlation_matrix')
plt.show()

In [ ]:
# --- Ranked correlations with 'completed' ---
# Extract the correlation of each engagement metric with the binary outcome
# and rank by absolute strength.
corr_with_completed = (
    corr_matrix['Completed']
    .drop('Completed')
    .reset_index()
)
corr_with_completed.columns = ['metric', 'correlation']
corr_with_completed['abs_correlation'] = corr_with_completed['correlation'].abs()
corr_with_completed = corr_with_completed.sort_values('abs_correlation', ascending=False)
corr_with_completed = corr_with_completed.drop(columns='abs_correlation')

print('=== Engagement Metrics Ranked by Correlation with Completion ===\n')
print(corr_with_completed.to_string(index=False))

> **Interpretazione:**
> - Le metriche di engagement sono **inter-correlate** — giorni attivi, click totali e decile di engagement tendono a muoversi insieme. Questo è previsto: studenti che cliccano di più tendono anche a cliccare in più giorni.
> - I correlati più forti con il completamento sono probabilmente **giorni attivi** e **click totali** — le misure di engagement più semplici sono anche le più predittive.
> - **Click per giorno attivo** (intensità) potrebbe avere una correlazione più debole con il completamento rispetto ai giorni attivi (frequenza), rafforzando il risultato della Sezione 6 che la costanza conta più dell'intensità.
> - Includere i ghost student (come zeri) rafforza tutte le correlazioni perché rappresentano il caso estremo: zero engagement, completamento prossimo allo zero.
>
> **Avvertenza:** Le correlazioni di Pearson catturano relazioni **lineari**. Pattern non lineari (es. rendimenti decrescenti ad alti livelli di engagement, o un effetto soglia) sarebbero sottostimati. L'analisi dei decili nella Sezione 4 affronta parzialmente questo problema esaminando la forma della curva dose-risposta.
>
> **Guardando avanti:** BQ2 (Notebook 04) formalizzerà queste associazioni con t-test, effect size (Cohen's d) e intervalli di confidenza — passando dalla correlazione alla valutazione di segnali azionabili.

## 10. Conclusioni e prossimi passi

### Cosa abbiamo imparato

1. **I ghost student sono il segnale più chiaro.** Una quota misurabile di iscrizioni ha zero attività VLE nei primi 28 giorni. Il loro tasso di completamento è prossimo allo zero. Sono il frutto più a portata di mano per gli interventi — hanno bisogno di *attivazione*, non di supporto accademico.

2. **L'engagement separa gli esiti su tutte le metriche.** I completers mostrano giorni attivi sostanzialmente più alti, più click totali e un ultimo giorno attivo più avanzato nella finestra di 28 giorni. La separazione è visibile in ogni grafico di distribuzione.

3. **Relazione dose-risposta.** Il tasso di completamento aumenta in modo monotono con il decile di engagement. I 2–3 decili più bassi sono segmenti a rischio estremo con tassi di completamento molto sotto la media.

4. **Divergenza precoce.** La traiettoria giornaliera mostra che il divario di engagement tra completers e non-completers si apre entro i primi 7–10 giorni. I non-completers sia cliccano meno *che* smettono di presentarsi prima — un doppio segnale di intensità e frequenza.

5. **Costanza batte intensità.** L'analisi tipologica rivela che la *frequenza* (numero di giorni attivi) predice il completamento più fortemente dell'*intensità* (click per giorno attivo). L'engagement steady supera quello binge.

6. **La diversità dei tipi di attività conta.** Studenti che usano più tipi di risorse completano a tassi più alti. L'ampiezza dell'engagement è un segnale positivo, sebbene parzialmente confuso con il volume totale.

7. **Variazione a livello di corso.** I livelli di engagement differiscono sostanzialmente tra i moduli, rafforzando la necessità di un'analisi course-aware. La normalizzazione all'interno del corso (decili di engagement) affronta questo problema.

8. **Correlati più forti.** Giorni attivi e click totali hanno la correlazione più alta con il completamento, seguiti dal decile di engagement e dall'ultimo giorno attivo. Questi sono candidati per un sistema di early warning.

### Cosa viene dopo

| Notebook | Business Question | Focus |
|----------|------------------|-------|
| **03** | BQ1 | Dove e quando abbandonano gli studenti? |
| **04** | BQ2 | Quali segnali comportamentali precoci predicono l'abbandono? |
| **05** | BQ3 | Demografica vs. comportamento — cosa predice meglio l'esito? |
| **06** | BQ4 | Come influiscono le caratteristiche del corso sulla retention? |
| **07** | BQ5 | Top 3 interventi azionabili |

---

**Riproducibilità:** Tutte le figure sono salvate in `reports/figures/`. Per riprodurre questo notebook, esegui prima `python -m run_pipeline`, poi esegui tutte le celle in ordine.

> **Dall'EDA all'analisi:** Questi due notebook EDA (01 + 02) hanno stabilito il profilo della popolazione, il panorama dell'engagement e le ipotesi chiave. Da qui in poi, i notebook rimanenti rispondono a specifiche business question con rigore statistico.
>
> Continua al **Notebook 03** (`03_bq1_dropout_timing.ipynb`) per BQ1: dove e quando abbandonano gli studenti?